# **CSE573 Group 17 Project 8 Movie Recommendations KGNN-RAG Chatbot**

Install Packages

In [1]:
!pip -q install pandas networkx matplotlib rapidfuzz \
  sentence-transformers faiss-cpu transformers accelerate gradio flask flask-cors
!npm install -g localtunnel > /dev/null 2>&1
!pip install flask flask-cors --ignore-installed blinker
!pip install sentence-transformers

  Using cached flask-3.1.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached flask_cors-6.0.1-py3-none-any.whl.metadata (5.3 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
  Using cached werkzeug-3.1.3-py3-none-any.whl.metadata (3.7 kB)
Using cached flask-3.1.2-py3-none-any.whl (103 kB)
Using cached flask_cors-6.0.1-py3-none-any.whl (13 kB)
Using cached blinker-1.9.0-py3-none-any.whl (8.5 kB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_2

Setup Imports & Utilities

In [2]:
# Standard library imports
import os, re, math, json, pickle, warnings, unicodedata, subprocess, threading, time
from collections import Counter
from dataclasses import dataclass, field

# Data science & graph imports
import pandas as pd
import numpy as np
import networkx as nx
import scipy.sparse as sp
import matplotlib.pyplot as plt

# ML & NLP imports
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rapidfuzz import process, fuzz
import faiss

# Web framework imports
from flask import Flask, request, jsonify
from flask_cors import CORS

# Configuration
warnings.filterwarnings("ignore")

# Model configuration
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GEN_MODEL = "google/flan-t5-small"

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Load MovieLens CSVs (movies, ratings, users)

In [21]:
DATA_DIR = "/content"

users   = pd.read_csv(os.path.join(DATA_DIR, "users.csv"))
movies  = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"))
ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))

# OPTIONAL: drop low-rated movies
min_avg_rating = 4.0
movie_avg_ratings = ratings.groupby('MovieID')['Rating'].mean()
good_movies = movie_avg_ratings[movie_avg_ratings > min_avg_rating].index
movies = movies[movies['MovieID'].isin(good_movies)]
ratings = ratings[ratings['MovieID'].isin(good_movies)]

print(f"Loaded {len(users):,} users, {len(movies):,} movies, {len(ratings):,} ratings")

Loaded 6,040 users, 370 movies, 236,170 ratings


Canonicalized & Flexible Title Index (helps with movie title lookups later)

In [22]:
# Flexible title index
ARTICLES = {"the","a","an"}

def _strip_year_suffix(s: str) -> str:
    return re.sub(r"\s*\(\d{4}\)\s*$", "", s.strip())

def _move_trailing_article(s: str) -> str:
    # "Matrix, The" -> "The Matrix"
    m = re.match(r"^(.*),\s*(The|A|An)$", s, flags=re.I)
    if not m:
        return s
    head, art = m.group(1), m.group(2)
    return f"{art} {head}"

def _normalize_tokens(s: str) -> str:
    # "Matrix, The" -> "the matrix"
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii","ignore").decode("ascii")
    s = s.lower()
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _variants(raw_title: str, year):
    base = _strip_year_suffix(raw_title)
    with_art = _move_trailing_article(base)
    toks1 = _normalize_tokens(base)
    toks2 = _normalize_tokens(with_art)
    vs = {toks1, toks2}
    if year and str(year).isdigit():
        y = str(int(year))
        vs |= {f"{toks1} {y}", f"{toks2} {y}"}
    # also drop leading article form: "the matrix" -> "matrix"
    for t in list(vs):
        words = [w for w in t.split() if w not in ARTICLES]
        if words:
            vs.add(" ".join(words))
    return vs

TITLE_VAR_INDEX = {}        # variant_key -> (node_id, display_title)
TITLE_KEYS = []             # list of variant keys for fuzzy search
TITLE_DISPLAY = {}          # node_id -> display_title

for _, row in movies.iterrows():
    node_id = f"movie_{int(row['MovieID'])}"
    raw = str(row["Title"])
    yr  = row.get("Year", None)
    for v in _variants(raw, yr):
        TITLE_VAR_INDEX.setdefault(v, (node_id, raw))
        TITLE_KEYS.append(v)
    TITLE_DISPLAY[node_id] = raw


Build Knowledge Graph (KGNN) - Same as before

In [23]:
# Build graph
G = nx.DiGraph()

# Track unique entities
age_buckets = set()
occupations = set()
genders = set()
genres = set()
year_decades = set()

# Occupation mapping
occupation_map = {
    0: "other", 1: "academic/educator", 2: "artist", 3: "clerical/admin",
    4: "college/grad student", 5: "customer service", 6: "doctor/health care",
    7: "executive/managerial", 8: "farmer", 9: "homemaker", 10: "K-12 student",
    11: "lawyer", 12: "programmer", 13: "retired", 14: "sales/marketing",
    15: "scientist", 16: "self-employed", 17: "technician/engineer",
    18: "tradesman/craftsman", 19: "unemployed", 20: "writer"
}

# adding nodes and relationships
for _, user in users.iterrows():
    user_id = f"user_{user['UserID']}"

    # Add user node
    G.add_node(user_id, ntype="user", user_id=int(user['UserID']))

    # Add and link gender
    gender = user['Gender']
    gender_id = f"gender_{gender}"
    if gender_id not in genders:
        G.add_node(gender_id, ntype="gender", name=gender)
        genders.add(gender_id)
    G.add_edge(user_id, gender_id, etype="is_gender")

    # Add and link age bucket
    age = user['Age']
    age_id = f"age_{age.replace('-', '_').replace('+', 'plus').replace('<', 'under')}"
    if age_id not in age_buckets:
        G.add_node(age_id, ntype="age_bucket", name=age)
        age_buckets.add(age_id)
    G.add_edge(user_id, age_id, etype="age_bucket")

    # Add and link occupation
    occ_code = int(user['Occupation'])
    occ_name = occupation_map.get(occ_code, "other")
    occ_id = f"occ_{occ_code}"
    if occ_id not in occupations:
        G.add_node(occ_id, ntype="occupation", name=occ_name, label=occ_name)
        occupations.add(occ_id)
    G.add_edge(user_id, occ_id, etype="has_occupation")


# add movie nodes
for _, movie in movies.iterrows():
    movie_id = f"movie_{movie['MovieID']}"

    # Add movie node
    G.add_node(movie_id,
               ntype="movie",
               movie_id=int(movie['MovieID']),
               title=movie['Title'],
               year=int(movie['Year']) if pd.notna(movie['Year']) else None)

    # Add and link genres
    if pd.notna(movie['Genres']):
        for genre in movie['Genres'].split('|'):
            genre = genre.strip()
            genre_id = f"genre_{genre.replace(' ', '_').replace('-', '_')}"
            if genre_id not in genres:
                G.add_node(genre_id, ntype="genre", name=genre, title=genre)
                genres.add(genre_id)
            G.add_edge(movie_id, genre_id, etype="is_genre")

    # Add and link year decade
    if pd.notna(movie['Year']):
        year = int(movie['Year'])
        decade = (year // 10) * 10
        year_id = f"year_{decade}s"
        if year_id not in year_decades:
            G.add_node(year_id, ntype="year", name=f"{decade}s", decade=decade)
            year_decades.add(year_id)
        G.add_edge(movie_id, year_id, etype="year_bucket")

# add ratings
for _, rating in ratings.iterrows():
    user_id = f"user_{rating['UserID']}"
    movie_id = f"movie_{rating['MovieID']}"
    rating_val = float(rating['Rating'])

    if user_id in G and movie_id in G:
        # 4-5 stars - rated highly
        if rating_val >= 4.0:
            G.add_edge(user_id, movie_id,
                       etype="rated_high",
                       rating=rating_val,
                       timestamp=int(rating['Timestamp']))
        # 1-2 stars - rated low
        elif rating_val <= 2.0:
            G.add_edge(user_id, movie_id,
                       etype="rated_low",
                       rating=rating_val,
                       timestamp=int(rating['Timestamp']))
        # 3 stars - neutral
        else:
            G.add_edge(user_id, movie_id,
                       etype="rated_medium",
                       rating=rating_val,
                       timestamp=int(rating['Timestamp']))


# Graph stats
print(f"\nGraph Structure:")
print(f"Total nodes: {G.number_of_nodes():,}")
print(f"Total edges: {G.number_of_edges():,}")
print(f"Graph density: {nx.density(G):.6f}")

node_type_counts = Counter(d['ntype'] for n, d in G.nodes(data=True))
print(f"\nNode Type Distribution:")
for ntype, count in node_type_counts.items():
    percentage = count / G.number_of_nodes() * 100
    print(f"{ntype.title()}: {count:,} ({percentage:.1f}%)")

edge_type_counts = Counter(d['etype'] for u, v, d in G.edges(data=True))
print(f"\nEdge Type Distribution:")
for etype, count in edge_type_counts.items():
    percentage = count / G.number_of_edges() * 100
    print(f"{etype}: {count:,} ({percentage:.1f}%)")


# visual of some connections

# Sample a few users and their connections
sample_users = list(users.head(3)['UserID'])
sample_nodes = set()
movie_nodes = []

for user_id in sample_users:
    user_node = f"user_{user_id}"
    if user_node in G:
        sample_nodes.add(user_node)
        # Add all neighbors (gender, age, occupation, rated movies)
        for neighbor in G.neighbors(user_node):
            # If it's a movie, limit to 30 total
            if G.nodes[neighbor]['ntype'] == 'movie':
                if len(movie_nodes) < 30:
                    movie_nodes.append(neighbor)
                    sample_nodes.add(neighbor)
                    # Add its genre and year too
                    for movie_neighbor in G.neighbors(neighbor):
                        sample_nodes.add(movie_neighbor)
            else:
                sample_nodes.add(neighbor)

# Create subgraph
subG = G.subgraph(sample_nodes)

# Visualize
fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Layout
pos = nx.spring_layout(subG, k=0.5, iterations=50, seed=42)

# Color map for node types - MUCH more distinct colors
color_map = {
    'user': '#1E88E5',      # Bright Blue
    'movie': '#D81B60',     # Pink/Magenta
    'genre': '#8E24AA',     # Deep Purple
    'year': '#FFB300',      # Amber/Gold
    'gender': '#00ACC1',    # Cyan
    'age_bucket': '#43A047', # Green
    'occupation': '#FB8C00'  # Orange
}

# Draw edges
nx.draw_networkx_edges(subG, pos, alpha=0.3, width=1.5, edge_color='gray',
                       arrows=True, arrowsize=15, ax=ax)

# Draw nodes by type
for ntype, color in color_map.items():
    nodes_of_type = [n for n in subG.nodes() if subG.nodes[n]['ntype'] == ntype]
    if nodes_of_type:
        nx.draw_networkx_nodes(subG, pos,
                             nodelist=nodes_of_type,
                             node_color=color,
                             node_size=800 if ntype == 'user' else 600,
                             alpha=0.8,
                             label=f"{ntype.replace('_', ' ').title()}",
                             ax=ax)

# Add labels
labels = {}
for node in subG.nodes():
    node_data = subG.nodes[node]
    ntype = node_data['ntype']

    if ntype == 'user':
        labels[node] = f"U{node_data['user_id']}"
    elif ntype == 'movie':
        title = node_data.get('title', 'Movie')[:20]
        labels[node] = title
    elif ntype in ['gender', 'age_bucket', 'occupation', 'genre', 'year']:
        labels[node] = node_data.get('name', node_data.get('title', node))[:15]

nx.draw_networkx_labels(subG, pos, labels, font_size=9, ax=ax)

ax.set_title(f"MovieLens Knowledge Graph Sample\n{len(sample_nodes)} nodes from {G.number_of_nodes():,} total",
            fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.axis('off')

plt.tight_layout()

# Save to file instead of displaying
output_file = '/content/movielens_graph.png'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"Graph visualization saved to: {output_file}")
plt.close()


# visualize ratings per movie

# Count ratings per movie
movie_rating_counts = ratings.groupby('MovieID').size().reset_index(name='num_ratings')

# Count how many movies have each number of ratings
# e.g., how many movies have 1 rating, 2 ratings, 3 ratings, etc.
rating_count_distribution = movie_rating_counts['num_ratings'].value_counts().sort_index()

# Create bucketed table
print("Rating Distrubution")
print(f"{'Rating Range':<20} {'Number of Movies':<20}")

# Helper function to count movies in a range
def count_in_range(start, end):
    return sum(rating_count_distribution.get(i, 0) for i in range(start, end + 1))

for start in range(1, 100, 10):
    end = start + 9
    count = count_in_range(start, end)
    print(f"{start}-{end:<17} {count:<20}")

for start in range(100, 500, 100):
    end = start + 99
    count = count_in_range(start, end)
    print(f"{start}-{end:<16} {count:<20}")

max_rating = movie_rating_counts['num_ratings'].max()
start = 500
while start <= max_rating:
    end = start + 499
    count = count_in_range(start, end)
    print(f"{start}-{end:<16} {count:<20}")
    start += 500


fig, ax = plt.subplots(1, 1, figsize=(14, 6))

x_values = rating_count_distribution.index.values
y_values = rating_count_distribution.values
width = 0.8

ax.bar(x_values, y_values, width=width, align='center',
       color='steelblue', edgecolor='black', alpha=0.7)
ax.set_xlabel('Number of Ratings', fontsize=12)
ax.set_ylabel('Number of Movies', fontsize=12)
ax.set_title('Distribution: Number of Movies per Rating Count', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_xlim(0, max(x_values) + 1)

plt.tight_layout()

# Save figure
plt.savefig('movie_rating_distribution.png', dpi=300, bbox_inches='tight')
print("\n✓ Rating distribution saved to: movie_rating_distribution.png")
plt.close()

# Print statistics
print(f"\nRating Distribution Statistics:")
print(f"Total movies: {len(movie_rating_counts):,}")
print(f"Average ratings per movie: {movie_rating_counts['num_ratings'].mean():.1f}")
print(f"Median ratings per movie: {movie_rating_counts['num_ratings'].median():.0f}")
print(f"Max ratings for a single movie: {movie_rating_counts['num_ratings'].max():,}")
print(f"Min ratings for a single movie: {movie_rating_counts['num_ratings'].min()}")

print(f"\nFinal Graph Structure:")
print(f"Total nodes: {G.number_of_nodes():,}")
print(f"Total edges: {G.number_of_edges():,}")
print(f"Graph density: {nx.density(G):.6f}")


Graph Structure:
Total nodes: 6,467
Total edges: 255,291
Graph density: 0.006105

Node Type Distribution:
User: 6,040 (93.4%)
Gender: 2 (0.0%)
Age_Bucket: 7 (0.1%)
Occupation: 21 (0.3%)
Movie: 370 (5.7%)
Genre: 18 (0.3%)
Year: 9 (0.1%)

Edge Type Distribution:
is_gender: 6,040 (2.4%)
age_bucket: 6,040 (2.4%)
has_occupation: 6,040 (2.4%)
rated_high: 192,220 (75.3%)
rated_medium: 33,799 (13.2%)
rated_low: 10,151 (4.0%)
is_genre: 631 (0.2%)
year_bucket: 370 (0.1%)
Graph visualization saved to: /content/movielens_graph.png
Rating Distrubution
Rating Range         Number of Movies    
1-10                29                  
11-20                5                   
21-30                6                   
31-40                1                   
41-50                6                   
51-60                3                   
61-70                9                   
71-80                4                   
81-90                4                   
91-100               6             

Build Docs from KGNN

In [24]:
# Precompute rating stats for cards
avg_by_movie = ratings.groupby("MovieID")["Rating"].mean().to_dict()
count_by_movie = ratings.groupby("MovieID")["Rating"].size().to_dict()

# Build docs
docs = [] # list of dicts: {id, kind, title, text, meta}
id2row = {} # map doc_id -> index
for n, data in G.nodes(data=True):
    ntype = data.get("ntype")
    if ntype == "movie":
        mid = data["movie_id"]
        title = data.get("title", f"movie {mid}")
        year = data.get("year")
        # neighbors: genres + decade
        movie_genres = [G.nodes[g]["name"] for g in G.neighbors(n) if G.nodes[g].get("ntype") == "genre"]
        decade = None
        for y in G.neighbors(n):
            if G.nodes[y].get("ntype") == "year":
                decade = G.nodes[y].get("name")
        num_r = int(count_by_movie.get(mid, 0))
        avg_r = float(avg_by_movie.get(mid, 0.0))
        text = f"{title} ({year}). Genres: {', '.join(movie_genres)}. Decade: {decade}. Popularity: {num_r} ratings. Average rating: {avg_r:.2f}."
        docs.append({
            "id": n, "kind": "movie", "title": title, "text": text,
            "meta": {"year": year, "genres": movie_genres, "decade": decade,
                     "num_ratings": num_r, "avg_rating": avg_r}
        })

# Create / load embedding model
emb_model = SentenceTransformer(EMB_MODEL)
embs = emb_model.encode([d["text"] for d in docs], show_progress_bar=True, normalize_embeddings=True).astype("float32")

dim = embs.shape[1]
index = faiss.IndexFlatIP(dim) # cosine (since normalized)
index.add(embs)

# Simple search helper (returns list of (idx, score))
def semantic_search(query: str, k: int = 25):
    q = emb_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q, k)
    return list(zip(idxs[0].tolist(), scores[0].tolist()))

# map index row to doc quickly
row2doc = docs


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Seed-aware RAG ranker used by chat/UI

If a seed title is detected (resolve_title_flexible), use CF neighbors (>=4★). Otherwise fall back to semantic search. Honors explicit genre/decade filters.
Returns: list of {title, year, decade, genres, avg, num}

In [25]:
def ask_structured(query: str, top_k: int = 5, forced_genres=None, forced_decade=None):
    # parse soft constraints from the free text you already extract
    cons = extract_constraints(query)
    if forced_genres:
        cons["genres"] = list({g.lower() for g in forced_genres})
    if forced_decade:
        cons["decade"] = forced_decade

    # resolve seed(s) from natural language
    seeds_resolved, oov_title, suggestion = resolve_title_flexible(query)
    seed_nodes = seeds_resolved

    candidates = {}

    # Seed path: CF neighbors dominate (with guards)
    if seed_nodes:
        seed_genres_union = set()
        seed_years = []
        for sn in seed_nodes:
            seed_genres_union |= _genres_for(sn)
            y = _year_for(sn)
            if y: seed_years.append(int(y))
        seed_year = int(np.median(seed_years)) if seed_years else None

        for sn in seed_nodes:
            try:
                mid = int(str(sn).split("_", 1)[1])
            except Exception:
                continue
            for node, cf_score, overlap in cf_similar_movies(mid, topk=400, min_overlap=15):
                if node in seed_nodes:
                    continue
                if not _passes_topic_guard(seed_genres_union, _genres_for(node), seed_year, _year_for(node),
                                           require_genre_overlap=True, max_year_gap=20):
                    continue
                prev = candidates.get(node, {"cf":0.0, "sem":0.0, "ov":0})
                prev["cf"] = max(prev["cf"], cf_score)
                prev["ov"] = max(prev["ov"], overlap)
                candidates[node] = prev

        # small semantic boost for tie-breakers
        try:
            sem_hits = semantic_search(query, k=300)
            sem_map = {row2doc[i]["id"]: sc for i, sc in sem_hits}
            for node in candidates.keys():
                candidates[node]["sem"] = float(sem_map.get(node, 0.0))
        except Exception:
            pass

    # No seed: semantic-only path
    else:
        sem_hits = semantic_search(query, k=600)
        for i, sc in sem_hits:
            node = row2doc[i]["id"]
            candidates[node] = {"cf":0.0, "sem": float(sc), "ov":0}

    # explicit filters
    def _passes_explicit(node):
        f = facts_for_movie(node)
        if cons["genres"]:
            cand = {g.lower() for g in (f.get("genres") or [])}
            if cand.isdisjoint(set(cons["genres"])): return False
        if cons["decade"]:
            if (f.get("decade") or "").lower() != cons["decade"].lower(): return False
        return True

    filtered = [n for n in candidates.keys()] if not (cons["genres"] or cons["decade"]) else \
               [n for n in candidates.keys() if _passes_explicit(n)]

    # final score (CF-heavy with seed, semantic-heavy otherwise) + slight popularity tie-break
    scored = []
    for n in filtered:
        f = facts_for_movie(n)
        pop = np.log1p(f.get("num", 0)) / 10.0
        cf  = candidates[n]["cf"]
        sem = candidates[n]["sem"]
        score = (0.9*cf + 0.08*sem + 0.02*pop) if seed_nodes else (0.85*sem + 0.15*pop)
        scored.append((n, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    top = scored[:top_k]

    # build UI-ready records
    out = []
    for n, _ in top:
        f = facts_for_movie(n)
        out.append({
            "title": f["title"],
            "year": f.get("year","?"),
            "genres": f.get("genres", []),
            "avg": round(f.get("avg", 0.0), 2),
            "num": f.get("num", 0),
            "decade": f.get("decade")
        })
    return out


Graph-Aware Context Recommendations



In [26]:
# Precompute for high ratings (>= 4.0)
movie_likers_high = {}
for u, m, d in G.edges(data=True):
    if d.get("etype") == "rated_high" and d.get("rating", 0) >= 4.0:
        movie_likers_high.setdefault(m, set()).add(u)

# Title lookup and fuzzy matching
title_to_node = {}
for n, data in G.nodes(data=True):
    if data.get("ntype") == "movie":
        title_to_node[data.get("title","").lower()] = n

def fuzzy_find_titles(text: str, limit: int = 5):
    choices = list(title_to_node.keys())
    matches = process.extract(text.lower(), choices, scorer=fuzz.WRatio, limit=limit)
    out = []
    for title_lc, score, _ in matches:
        node = title_to_node[title_lc]
        out.append((G.nodes[node].get("title", title_lc), node, int(score)))
    return out

def jaccard(A: set, B: set) -> float:
    if not A or not B: return 0.0
    inter = len(A & B); uni = len(A | B)
    return inter / uni if uni else 0.0

def similar_by_likers(seed_nodes, k=50):
    seed_likers = set().union(*(movie_likers_high.get(n, set()) for n in seed_nodes))
    scores = {}
    for m, likers in movie_likers_high.items():
        if m in seed_nodes:
            continue
        scores[m] = jaccard(seed_likers, likers)
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:k]

# Constraint extraction: genres & decades ("90s", "1990s", etc.)
all_genres = {G.nodes[n]["name"].lower() for n, d in G.nodes(data=True) if d.get("ntype") == "genre"}

def extract_constraints(text: str):
    tg = set()
    tl = text.lower()
    for g in all_genres:
        if re.search(rf"\b{re.escape(g)}\b", tl):
            tg.add(g)
    # Decade patterns: 1990s / 2000s / 80s / 90s
    decade = None
    m = re.search(r"(?:19|20)?\d0s", tl)
    if m:
        s = m.group(0)
        # ex. "90s" = "1990s"
        if re.fullmatch(r"\d0s", s):
            decade = f"19{s}"
        else:
            decade = s
    return {"genres": list(tg), "decade": decade}

User Demographic Parsing

In [27]:
def age_value_to_bucket(val: int) -> int:
    if val < 18:
        return 1
    elif val < 25:
        return 18
    elif val < 35:
        return 25
    elif val < 45:
        return 35
    elif val < 50:
        return 45
    elif val < 56:
        return 50
    else:
        return 56


def parse_demographics(text: str):
    text_l = text.lower()

    # gender detection
    gender = None

    if re.search(r"\b(male|man|males|men)\b", text_l):
        gender = "male"

    if re.search(r"\b(female|woman|women|females)\b", text_l):
        gender = "female"

    # occupation
    occupation_synonyms = {
        0: ["other"],  # fallback
        1: ["teacher", "professor", "instructor", "lecturer", "educator", "academic"],
        2: ["artist", "painter", "designer", "illustrator", "musician", "singer", "photographer", "creative"],
        3: ["admin", "administrator", "secretary", "office worker", "clerical"],
        4: ["college student", "university student", "grad student", "graduate student", "undergrad", "student"],
        5: ["customer service", "csr", "call center"],
        6: ["doctor", "nurse", "therapist", "healthcare worker", "medical professional", "health care"],
        7: ["manager", "executive", "ceo", "cto", "director", "managerial"],
        8: ["farmer", "agriculture", "agricultural"],
        9: ["homemaker", "stay at home mom", "stay at home dad", "stay-at-home"],
        10: ["high school student", "middle school student", "k-12 student", "elementary student", "school kid"],
        11: ["lawyer", "attorney"],
        12: ["programmer", "developer", "software engineer", "coder", "dev"],
        13: ["retired", "pensioner", "senior citizen"],
        14: ["sales", "salesperson", "marketing", "marketer"],
        15: ["scientist", "researcher"],
        16: ["self-employed", "freelancer"],
        17: ["technician", "mechanic", "engineer"],
        18: ["carpenter", "plumber", "electrician", "craftsman", "tradesman"],
        19: ["unemployed", "jobless", "out of work"],
        20: ["writer", "author", "journalist"]
    }

    occ = None

    # "student"/"students" without qualifier = college student (4)
    # b/c there are two groups of students (4 & 11)
    if re.search(r"\bstudent[s]?\b", text_l):
        occ = 4

    # Search
    if occ is None:
        for code, keywords in occupation_synonyms.items():
            for kw in keywords:
                if kw in text_l:
                    occ = code
                    break
            if occ is not None:
                break

    # age detection
    age_bucket = None

    # Explicit under 18 variants, due to issues
    if ("under 18" in text_l) or ("under18" in text_l) or ("<18" in text_l) or ("younger than 18" in text_l):
        age_bucket = 1

    if age_bucket is None:
        bucket_phrases = {
            "under 18": 1,
            "18-24": 18,
            "25-34": 25,
            "35-44": 35,
            "45-49": 45,
            "50-55": 50,
            "56+": 56
        }
        for phrase, code in bucket_phrases.items():
            if phrase in text_l:
                age_bucket = code
                break

    # Phrases like "50+", "35 and older"
    if age_bucket is None:
        m = re.search(r"(\d+)\s*\+|\b(\d+)\s*and older\b", text_l)
        if m:
            val = int(m.group(1) or m.group(2))
            age_bucket = age_value_to_bucket(val)

    # Simple numeric ages: "18 year olds", "23 yo", "I am 40"
    if age_bucket is None:
        m = re.search(r"\b(\d{1,2})\b", text_l)
        if m:
            val = int(m.group(1))
            age_bucket = age_value_to_bucket(val)


    return {
        "gender": gender,
        "age_bucket": age_bucket,
        "occupation": occ
    }


Item-Item CF

In [28]:
_hi = ratings.loc[ratings["Rating"] >= 4, ["UserID","MovieID"]].copy()
_uids = np.sort(_hi["UserID"].unique())
_mids = np.sort(movies["MovieID"].unique())

_uid2i = {u:i for i,u in enumerate(_uids)}
_mid2i = {m:i for i,m in enumerate(_mids)}
_i2mid = np.array(_mids)

row = _hi["MovieID"].map(_mid2i).values
col = _hi["UserID"].map(_uid2i).values
data = np.ones_like(row, dtype=np.float32)

# M x U (CSR)
R = sp.csr_matrix((data, (row, col)), shape=(len(_mids), len(_uids))).tocsr()

# L2 normalize rows
row_sq_sum = np.asarray(R.power(2).sum(axis=1)).ravel()
row_norm = np.sqrt(np.maximum(row_sq_sum, 1.0))
R_norm = R.multiply(1.0 / row_norm[:, None]).tocsr()

# Boolean for shared-user overlap
R_bool = R.astype(bool).tocsr().astype(np.int8)

def cf_similar_movies(mid: int, topk: int = 200, min_overlap: int = 20):
    i = _mid2i.get(mid, None)
    if i is None: return []
    ri_norm = R_norm.getrow(i)
    sims = (ri_norm @ R_norm.T).toarray().ravel()
    sims[i] = 0.0
    ri_bool = R_bool.getrow(i)
    ov = (ri_bool @ R_bool.T).toarray().ravel()

    mask = (ov >= min_overlap) & (sims > 0)
    idxs = np.where(mask)[0]
    if idxs.size == 0: return []
    order = idxs[np.argsort(-sims[idxs])]
    return [(f"movie_{int(_i2mid[j])}", float(sims[j]), int(ov[j])) for j in order[:topk]]


Topic Guards (keep neighbors on topic)

In [29]:
def _genres_for(node_id):
    f = facts_for_movie(node_id)
    return set((f.get("genres") or []))

def _year_for(node_id):
    f = facts_for_movie(node_id)
    return f.get("year", None)

def _passes_topic_guard(seed_genres, cand_genres, seed_year, cand_year,
                        require_genre_overlap=True, max_year_gap=20):
    if require_genre_overlap and seed_genres and cand_genres:
        if seed_genres.isdisjoint(cand_genres):
            return False
    if seed_year and cand_year:
        if abs(int(seed_year) - int(cand_year)) > max_year_gap:
            return False
    return True


Movie Stats (movie fact helper + stats)

In [30]:
_movie_stats = ratings.groupby('MovieID').Rating.agg(['count','mean']).rename(
    columns={'count':'num','mean':'avg'}
)
MOVIE_STATS = _movie_stats.to_dict(orient='index')

def facts_for_movie(node_id: str):
    mid = int(str(node_id).split('_', 1)[1])
    mrow = movies.loc[movies['MovieID'] == mid]
    if mrow.empty:
        return {'title':'?', 'year':None, 'decade':None, 'genres':[], 'avg':0.0, 'num':0}
    m = mrow.iloc[0]
    year = int(m['Year']) if pd.notna(m['Year']) else None
    decade = f"{(year//10)*10}s" if year else None
    genres = [g.strip() for g in str(m['Genres']).split('|')] if pd.notna(m['Genres']) else []
    st = MOVIE_STATS.get(mid, {'num':0, 'avg':0.0})
    return {
        'title': str(m['Title']),
        'year': year,
        'decade': decade,
        'genres': genres,
        'avg': float(st['avg']) if st['num'] else 0.0,
        'num': int(st['num']) if st['num'] else 0,
    }


Demographic Movie Recommendations

In [31]:
def recommend_by_demographics(gender=None, age_bucket=None, occupation=None, top_k=10):

    matching_users = []

    # Loop through all users
    for u, data in G.nodes(data=True):
        if data.get("ntype") != "user":
            continue

        ok = True

        # gender
        if gender is not None:
            has_g = False
            for nbr in G.neighbors(u):
                if G.nodes[nbr].get("ntype") == "gender":
                    g_name = G.nodes[nbr]["name"].lower().strip()  # stored as "Male"/"Female"
                    if g_name == gender.lower().strip():
                        has_g = True
            if not has_g:
                ok = False

        # age
        if ok and age_bucket is not None:
            bucket_to_name = {
                1:  "under 18",
                18: "18-24",
                25: "25-34",
                35: "35-44",
                45: "45-49",
                50: "50-55",
                56: "56+"
            }

            bucket_name = bucket_to_name.get(age_bucket)
            if bucket_name is None:
                ok = False
            else:
                # name to node
                age_node = (
                    "age_" +
                    bucket_name.replace("-", "_")
                               .replace("+", "plus")
                               .replace("<", "under")
                )

                has_age = False
                for nbr in G.neighbors(u):
                    if nbr == age_node:
                        has_age = True
                        break
                if not has_age:
                    ok = False


        # occupation
        if ok and occupation is not None:
            has_occ = False
            for nbr in G.neighbors(u):
                if G.nodes[nbr].get("ntype") == "occupation":
                    if G.nodes[nbr]["label"].lower().strip() == occupation_map[occupation].lower().strip():
                        has_occ = True
            if not has_occ:
                ok = False

        if ok:
            matching_users.append(u)

    # If no users matched = return empty list + user_count = 0
    if not matching_users:
        return [], 0

    # movies from matching users
    movie_scores = {}

    for u in matching_users:
        for nbr in G.neighbors(u):
            edge = G.get_edge_data(u, nbr)
            if edge and edge.get("etype") == "rated_high":
                movie_scores[nbr] = movie_scores.get(nbr, 0) + edge["rating"]

    if not movie_scores:
        return [], len(matching_users)

    # Sort movies by total rating score
    ranked = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)

    # Format output
    out = []
    for m, score in ranked[:top_k]:
        f = facts_for_movie(m)
        out.append({
            "title": f["title"],
            "year": f["year"],
            "genres": f["genres"],
            "avg": f["avg"],
            "num": f["num"],
            "decade": f["decade"]
        })

    return out, len(matching_users)


In [32]:
[n for n,d in G.nodes(data=True) if d.get("ntype")=="gender"]
[(n, d["name"]) for n,d in G.nodes(data=True) if d.get("ntype")=="gender"]


[('gender_Female', 'Female'), ('gender_Male', 'Male')]

Flexible resolver: accepts natural phrasing, rejects obvious typos with thresholds

In [33]:
try:
    from rapidfuzz import process, fuzz
except Exception:
    !pip -q install rapidfuzz
    from rapidfuzz import process, fuzz

_title_ref_re = re.compile(
    r"(?:like|similar to|if i (?:liked|love[dn]?))\s+\"?(.+?)\"?$",
    re.IGNORECASE
)

ACCEPT_THRESH = 93 # accept automatically if score >=
SUGGEST_THRESH = 86 # if in [SUGGEST, ACCEPT), ask user to confirm below SUGGEST -> treat as OOV

def _canon_query(s: str) -> str:
    s = _strip_year_suffix(s)
    s = _move_trailing_article(s)
    return _normalize_tokens(s)

def resolve_title_flexible(user_text: str):
    text = user_text.strip()

    # Check for explicit "like/similar to X" pattern
    m = _title_ref_re.search(text)
    if m:
        # User explicitly referenced a title
        cand = m.group(1)
        qkey = _canon_query(cand)

        # Exact variant hit?
        hit = TITLE_VAR_INDEX.get(qkey)
        if hit:
            return [hit[0]], None, None

        # Fuzzy search
        best = process.extractOne(qkey, TITLE_KEYS, scorer=fuzz.token_set_ratio)
        if not best:
            return [], cand, None
        key, score, _ = best
        node, disp = TITLE_VAR_INDEX[key]

        if score >= ACCEPT_THRESH:
            return [node], None, None
        elif score >= SUGGEST_THRESH:
            return [], cand, disp
        else:
            return [], cand, None

    # Only try implicit title matching if query is short and title-like
    words = text.split()
    if len(words) <= 6 and not any(kw in text.lower() for kw in ['movies', 'films', 'for', 'recommend', 'like', 'similar']):
        # Might be a bare title like "The Matrix" or "Toy Story"
        qkey = _canon_query(text)

        # Exact variant hit?
        hit = TITLE_VAR_INDEX.get(qkey)
        if hit:
            return [hit[0]], None, None

        # Fuzzy search with HIGH threshold to avoid false matches
        best = process.extractOne(qkey, TITLE_KEYS, scorer=fuzz.token_set_ratio)
        if best:
            key, score, _ = best
            if score >= 95:  # Very high threshold for implicit matching
                node, disp = TITLE_VAR_INDEX[key]
                return [node], None, None

    # else no title matching - return empty
    return [], None, None


Conversational State + tiny NLU (regex + fuzzy titles)

In [34]:
@dataclass
class ConversationState:
    preferred_genres: set = field(default_factory=set)
    disliked_genres: set = field(default_factory=set)
    decade: str | None = None
    top_k: int = 5
    seen_titles: set = field(default_factory=set)
    last_query: str = "" # last free-form user utterance
    last_filters: dict = field(default_factory=dict) # snapshot of filters we used

# quick helpers
def _find_genres_in_text(text: str):
    tl = text.lower()
    hits = []
    for g in all_genres:
        if re.search(rf"\b{re.escape(g)}\b", tl):
            hits.append(g)
    return hits

def _find_decade_in_text(text: str):
    tl = text.lower()
    m = re.search(r"(?:19|20)?\d0s", tl) # 80s, 1990s, 2000s
    if not m: return None
    s = m.group(0)
    if re.fullmatch(r"\d0s", s): # "90s" -> "1990s" best-effort
        return f"19{s}"
    return s

def _find_k_in_text(text: str, default_k: int = 5):
    m = re.search(r"(?:top|give|show|recommend)\s+(\d{1,2})", text.lower())
    if m:
        try:
            k = int(m.group(1))
            return max(1, min(20, k))
        except:
            pass
    return default_k

def update_state_from_text(state: ConversationState, text: str):
    # likes / dislikes
    tl = text.lower()
    like_hits = _find_genres_in_text(text)
    # "no X", "not X", "no horror", "no rom-coms"
    dislike_hits = []
    for g in all_genres:
        if re.search(rf"\b(no|not)\s+{re.escape(g)}\b", tl):
            dislike_hits.append(g)

    state.preferred_genres |= set(like_hits)
    state.disliked_genres |= set(dislike_hits)

    # decade and k
    dec = _find_decade_in_text(text)
    if dec: state.decade = dec
    state.top_k = _find_k_in_text(text, default_k=state.top_k)

    return []

# Genres for a few movies
print("Genres in all_genres:", sorted(all_genres))
print("\nSample movie genres from facts_for_movie:")
for i in range(5):
    node = f"movie_{movies.iloc[i]['MovieID']}"
    f = facts_for_movie(node)
    print(f"{f['title']}: {f['genres']}")


Genres in all_genres: ['action', 'adventure', 'animation', "children's", 'comedy', 'crime', 'documentary', 'drama', 'fantasy', 'film-noir', 'horror', 'musical', 'mystery', 'romance', 'sci-fi', 'thriller', 'war', 'western']

Sample movie genres from facts_for_movie:
Toy Story: ['Animation', "Children's", 'Comedy']
Sense and Sensibility: ['Drama', 'Romance']
Persuasion: ['Romance']
City of Lost Children, The: ['Adventure', 'Sci-Fi']
Seven (Se7en): ['Crime', 'Thriller']


Tiny Local Generator (load)

In [35]:
tok = AutoTokenizer.from_pretrained(GEN_MODEL)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL)

def summarize_with_t5(prompt: str, max_new_tokens: int = 128) -> str:
    inputs = tok(prompt, return_tensors="pt", truncation=True)
    out = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)


Conversational Engine

Unified recommendation engine that considers ALL constraints:
    - Demographics: gender, age_bucket, occupation
    - Content: genres, decade
    - Context: seed movies, preferences

In [36]:
def conversational_recommend(message: str, state: ConversationState):

    # 1. Extract Constraints
    # Demographics
    demo = parse_demographics(message)
    gender = demo.get("gender")
    age_bucket = demo.get("age_bucket")
    occupation = demo.get("occupation")

    # Content constraints (genre, decade)
    cons_now = extract_constraints(message)
    forced_genres = set(state.preferred_genres)
    if cons_now["genres"]:
        forced_genres |= {g.lower() for g in cons_now["genres"]}
    forced_genres -= set(state.disliked_genres)
    forced_decade = cons_now["decade"] or state.decade

    # Seed titles
    seeds_resolved, oov_title, suggestion = resolve_title_flexible(message)
    seeds_from_text = update_state_from_text(state, message)
    seed_nodes = list({*seeds_from_text, *seeds_resolved})

    # Handle OOV titles
    if oov_title is not None:
        if suggestion:
            reply = (f"Sorry — I don't have \"{oov_title}\" in this dataset (MovieLens-1M), "
                     f"so I can't do title-based similarity.\n\n"
                     f"Did you mean **{suggestion}**? If so, try: `like \"{suggestion}\"`.\n"
                     f"Or tell me genres/decade instead.")
        else:
            reply = (f"Sorry — I don't have \"{oov_title}\" in this dataset (MovieLens-1M), "
                     f"so I can't do title-based similarity. "
                     f"Try another seed that's here (e.g., `like \"The Matrix (1999)\"`, "
                     f"or tell me genres/decade instead.")
        return reply, "", state

    # 2. Retriaval Strat & Candidates
    has_demographics = (gender is not None or age_bucket is not None or occupation is not None)
    has_content_filters = (bool(forced_genres) or forced_decade is not None)
    has_seeds = bool(seed_nodes)

    if has_demographics:
        # Demographic
        # Get all movies liked by matching users, then filter by content
        matching_users = []

        for u, data in G.nodes(data=True):
            if data.get("ntype") != "user":
                continue

            user_ok = True

            # Gender filter
            if gender is not None:
                has_g = False
                for nbr in G.neighbors(u):
                    if G.nodes[nbr].get("ntype") == "gender":
                        if G.nodes[nbr]["name"].lower().strip() == gender.lower().strip():
                            has_g = True
                            break
                if not has_g:
                    user_ok = False

            # Age filter
            if user_ok and age_bucket is not None:
                bucket_to_name = {
                    1: "under 18", 18: "18-24", 25: "25-34",
                    35: "35-44", 45: "45-49", 50: "50-55", 56: "56+"
                }
                bucket_name = bucket_to_name.get(age_bucket)
                if bucket_name:
                    age_node = "age_" + bucket_name.replace("-", "_").replace("+", "plus").replace("<", "under")
                    has_age = any(nbr == age_node for nbr in G.neighbors(u))
                    if not has_age:
                        user_ok = False
                else:
                    user_ok = False

            # Occupation filter
            if user_ok and occupation is not None:
                has_occ = False
                for nbr in G.neighbors(u):
                    if G.nodes[nbr].get("ntype") == "occupation":
                        if G.nodes[nbr]["label"].lower().strip() == occupation_map[occupation].lower().strip():
                            has_occ = True
                            break
                if not has_occ:
                    user_ok = False

            if user_ok:
                matching_users.append(u)

        # Collect movies from matching users
        movie_scores = {}
        for u in matching_users:
            for nbr in G.neighbors(u):
                edge = G.get_edge_data(u, nbr)
                if edge and edge.get("etype") == "rated_high":
                    movie_scores[nbr] = movie_scores.get(nbr, 0) + edge["rating"]

        candidates = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)

    else:
        # content
        if seed_nodes:
            # Use CF similarity
            seed_genres_union = set()
            seed_years = []
            for sn in seed_nodes:
                seed_genres_union |= _genres_for(sn)
                y = _year_for(sn)
                if y: seed_years.append(int(y))
            seed_year = int(np.median(seed_years)) if seed_years else None

            candidates_dict = {}
            for sn in seed_nodes:
                try:
                    mid = int(str(sn).split("_", 1)[1])
                except Exception:
                    continue
                for node, cf_score, overlap in cf_similar_movies(mid, topk=400, min_overlap=15):
                    if node in seed_nodes:
                        continue
                    if not _passes_topic_guard(seed_genres_union, _genres_for(node),
                                               seed_year, _year_for(node),
                                               require_genre_overlap=True, max_year_gap=20):
                        continue
                    prev = candidates_dict.get(node, {"cf": 0.0, "sem": 0.0})
                    prev["cf"] = max(prev["cf"], cf_score)
                    candidates_dict[node] = prev

            # Small semantic boost
            try:
                sem_hits = semantic_search(message, k=300)
                sem_map = {row2doc[i]["id"]: sc for i, sc in sem_hits}
                for node in candidates_dict.keys():
                    candidates_dict[node]["sem"] = float(sem_map.get(node, 0.0))
            except Exception:
                pass

            # Score: CF-heavy with popularity tie-break
            scored = []
            for n in candidates_dict.keys():
                f = facts_for_movie(n)
                pop = np.log1p(f.get("num", 0)) / 10.0
                cf = candidates_dict[n]["cf"]
                sem = candidates_dict[n]["sem"]
                score = 0.9*cf + 0.08*sem + 0.02*pop
                scored.append((n, score))

            candidates = sorted(scored, key=lambda x: x[1], reverse=True)

        else:
            # Pure semantic search
            sem_hits = semantic_search(message, k=600)
            candidates = [(row2doc[i]["id"], sc) for i, sc in sem_hits]

    # 3. content filters
    def passes_content_filters(node):
        f = facts_for_movie(node)

        # Genre filter
        if forced_genres:
            movie_genres_lower = [g.lower() for g in f.get("genres", [])]
            if not any(g.lower() in movie_genres_lower for g in forced_genres):
                return False

        # Decade filter
        if forced_decade:
            if f.get("decade", "").lower() != forced_decade.lower():
                return False

        return True

    filtered_candidates = [(n, s) for n, s in candidates if passes_content_filters(n)]

    # 4. final ranking
    # Add popularity boost and avoid repeats
    def final_score(node, base):
        f = facts_for_movie(node)
        pop = np.log1p(f.get("num", 0)) / 10.0
        return base + 0.05 * pop

    scored_final = [(n, final_score(n, s)) for n, s in filtered_candidates]
    scored_final.sort(key=lambda x: x[1], reverse=True)

    # Remove already seen movies
    top_candidates = [n for n, s in scored_final if TITLE_DISPLAY.get(n, "") not in state.seen_titles]

    # Get top K
    top_nodes = top_candidates[:state.top_k]

    # 5. response
    recs = []
    for node in top_nodes:
        f = facts_for_movie(node)
        recs.append({
            "title": f["title"],
            "year": f.get("year", "?"),
            "genres": f.get("genres", []),
            "avg": round(f.get("avg", 0.0), 2),
            "num": f.get("num", 0),
            "decade": f.get("decade")
        })

    # Update state
    state.seen_titles.update({r["title"] for r in recs})
    state.last_query = message

    # Build description of what was applied
    desc_parts = []

    if has_demographics:
        demo_parts = []
        if gender: demo_parts.append(gender.capitalize())
        if age_bucket: demo_parts.append(f"Age {age_bucket}")
        if occupation is not None: demo_parts.append(occupation_map[occupation].capitalize())

        if has_demographics and 'matching_users' in locals():
            desc_parts.append(f"**{len(matching_users)} users** matching: {', '.join(demo_parts)}")
        else:
            desc_parts.append(f"Users matching: {', '.join(demo_parts)}")

    if forced_genres:
        desc_parts.append(f"**Genre**: {', '.join(sorted(forced_genres))}")

    if forced_decade:
        desc_parts.append(f"**Decade**: {forced_decade}")

    if seed_nodes:
        seed_titles = [TITLE_DISPLAY.get(n, n) for n in seed_nodes[:2]]
        desc_parts.append(f"**Similar to**: {', '.join(seed_titles)}")

    if desc_parts:
            reply_text = f"Using: {' | '.join(desc_parts)}\n\nHere are {len(recs)} recommendations:"
    else:
        reply_text = f"Here are {len(recs)} recommendations based on your query:"

    return reply_text, recs, state


Chatbot - Simple Flask API + Localtunnel

In [37]:
app = Flask(__name__)
CORS(app)

# test with
@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "healthy", "service": "MovieLens API"})

# use
@app.route('/api/recommend', methods=['POST'])
def recommend():
    try:
        data = request.json
        query = data.get('query', '')
        session_state = data.get('session_state', '{}')

        # Parse session state
        if session_state and session_state != "{}":
            state_dict = json.loads(session_state)
            state = ConversationState(
                preferred_genres=set(state_dict.get("preferred_genres", [])),
                disliked_genres=set(state_dict.get("disliked_genres", [])),
                decade=state_dict.get("decade"),
                top_k=state_dict.get("top_k", 5),
                seen_titles=set(state_dict.get("seen_titles", [])),
                last_query=state_dict.get("last_query", ""),
                last_filters=state_dict.get("last_filters", {})
            )
        else:
            state = ConversationState()

        # Get recommendations
        reply_text, recs_from_conv, updated_state = conversational_recommend(query, state)
        forced_genres = list(updated_state.preferred_genres - updated_state.disliked_genres)
        recs = ask_structured(
            query,
            top_k=updated_state.top_k,
            forced_genres=forced_genres if forced_genres else None,
            forced_decade=updated_state.decade
        )

        # Build response
        state_out = {
            "preferred_genres": list(updated_state.preferred_genres),
            "disliked_genres": list(updated_state.disliked_genres),
            "decade": updated_state.decade,
            "top_k": updated_state.top_k,
            "seen_titles": list(updated_state.seen_titles),
            "last_query": updated_state.last_query,
            "last_filters": updated_state.last_filters
        }

        return jsonify({
            "success": True,
            "message": reply_text,
            "recommendations": recs,
            "session_state": json.dumps(state_out)
        })

    except Exception as e:
        import traceback
        return jsonify({
            "success": False,
            "error": str(e),
            "traceback": traceback.format_exc()
        }), 500

# Start Flask
def run_flask():
    app.run(host='0.0.0.0', port=5001, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

# Get tunnel password
result = subprocess.run(['curl', '-s', 'https://loca.lt/mytunnelpassword'],
                       capture_output=True, text=True, timeout=10)
tunnel_password = result.stdout.strip()

# Start localtunnel
localtunnel_url = None
def run_lt():
    global localtunnel_url
    proc = subprocess.Popen(['lt', '--port', '5001'],
                           stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT,
                           text=True)
    for line in iter(proc.stdout.readline, ''):
        if 'your url is:' in line.lower():
            import re
            match = re.search(r'https://[a-z0-9-]+\.loca\.lt', line)
            if match:
                localtunnel_url = match.group(0)

lt_thread = threading.Thread(target=run_lt, daemon=True)
lt_thread.start()
time.sleep(8)

# Display info
print(f"API URL: {localtunnel_url if localtunnel_url else 'Loading...'}")
print(f"Password: {tunnel_password}")

# Keep alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nStopped")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5001 is in use by another program. Either identify and stop that program, or start the server with a different port.


API URL: https://young-lemons-hide.loca.lt
Password: 34.82.113.89


INFO:werkzeug:127.0.0.1 - - [26/Nov/2025 00:06:13] "GET / HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [26/Nov/2025 00:06:14] "GET /favicon.ico HTTP/1.1" 404 -



Stopped
